In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.font_manager as fm
import matplotlib

font_path = 'C:\\Windows\\Fonts\\gulim.ttc'
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc('font', family=font)

In [ ]:
# 파일 읽기 / 날씨 데이터와 전력량 데이터 (2015~2024)
weather_df = pd.read_csv('./data/weather_data_201501_202412.csv', encoding='cp949')
electric_df = pd.read_csv('./data/elect_2015_2024.csv',encoding='cp949')

In [ ]:
# unamed 컬럼 제외
electric_df = electric_df.drop(electric_df.columns[0],axis=1)

# 일시에서 월(month)뽑기
weather_df['월'] = pd.to_datetime(weather_df['일시']).dt.month.astype(int)

# 일시, 년(year)으로 교체
weather_df['일시'] =  pd.to_datetime(weather_df['일시']).dt.strftime('%Y').astype(int)

# 결측값 0으로 채우기 
weather_df['최심적설(cm)'] = weather_df['최심적설(cm)'].fillna(0)

# 일시를 연도로 바꾸기 - 뒤에 전력량과 맞춰주기 위하여
weather_df = weather_df.rename(columns={'일시':'연도'})

weather_df


In [ ]:
# 서울시 주택용 전력량 뽑기
seoul_electric_df = electric_df[electric_df['시도'] == '서울특별시']



dfs = []

# 각 연도별로 구별 계약종별 데이터 넣기
for y in range(2014,2025):
      seoul_electric_df_year = seoul_electric_df[seoul_electric_df['연도'] == y]

      seoul_gu_25 = [
          "종로구", "중구", "용산구", "성동구", "동대문구",
          "성북구", "도봉구", "은평구", "서대문구", "마포구",
          "강서구", "구로구", "영등포구", "동작구", "관악구",
          "강남구", "강동구", "송파구", "중랑구", "노원구",
          "서초구", "양천구", "광진구", "강북구", "금천구"
      ]


        # 구별 데이터 선별
      for gu in seoul_gu_25:
            seoul_electric_df_year_gu = seoul_electric_df_year[seoul_electric_df_year['시군구'] == gu]


            dt_type = ['주택용','일반용','교육용','산업용','농사용','가로등','심 야']


            # 계약별 데이터 선별
            for t in dt_type:
                  seoul_electric_df_year_gu_type = seoul_electric_df_year_gu[seoul_electric_df_year_gu["계약종별"] == t]

                  id_cols = ["연도","시도", "시군구", "계약종별"]
                  month_cols = [ f"{i}월" for i in range(1, 13)]

                    # 멜트해서 id_cols 남기기
                  df_melt = (
                      seoul_electric_df_year_gu_type
                      .reset_index(drop=True)
                      .melt(
                          id_vars=id_cols,
                          value_vars=month_cols,
                          var_name="월",
                          value_name="전력량"
                      )
                  )
                  # 월 숫자만 남기기
                  df_melt["월"] = df_melt["월"].str.replace("월", "").astype(int)
                  dfs.append(df_melt)

# 연도별 구별 월별 데이터 합치기
electric_melt_all = pd.concat(dfs, ignore_index=True)

electric_melt_all

In [ ]:
# # 2015년 서울시 날씨값 뽑기
# weather_seoul_df = weather_df[(weather_df['지점명'] == '서울') & (weather_df['연도'] == 2015)]
# weather_seoul_df


In [ ]:
# 서울시 날씨 데이터 뽑기
weather_seoul_df = weather_df[weather_df['지점명'] == '서울']
weather_seoul_df

In [ ]:
# 날씨값에서 지점, 지점명, 빼기 (중복 or 불필요)
weather_seoul_df_slice = weather_seoul_df.iloc[:,2:]
weather_seoul_df_slice

In [ ]:
# 날씨, 전력량 조인
df = electric_melt_all.merge(weather_seoul_df_slice, on=['연도','월'])
# df_final_2015[df_final_2015['계약종별'] == '심야']

# 날짜로 연도, 월 병합하기
df['날짜'] = df['연도'].astype(str) + '-' + df['월'].astype(str).str.zfill(2)

# 날짜 컬럼 인덱스로 돌리기
df_final = df.set_index('날짜').drop(columns=['연도','월'])

In [ ]:
# 최종 데이터

t = df_final['평균기온(°C)']
df_final['HDD'] = (18 - t).clip(lower=0)
df_final['CDD'] = (t - 24).clip(lower=0)

df_final


In [ ]:
# 구별, 월별, 종별의 전력량과 날씨 데이터 상관관계 heatmap
for idx,t in enumerate(dt_type):
    df_jongro = df_final[(df_final['계약종별']==t)]
    corr_matrix = df_jongro.corr(numeric_only=True)
    corr_matrix
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='YlGnBu_r')
    print(f'계약종별: {dt_type[idx]}의 상관관계')
    plt.show()

In [ ]:
# 구별, 월별, 종별의 전력량과 날씨 데이터 상관관계 heatmap.version2
for idx,t in enumerate(dt_type):
    df_jongro = df_final[(df_final['계약종별']==t)]


    num_df = df_jongro.select_dtypes(include='number')
    
    corr = num_df.corr()
    corr_power = corr[['전력량']].sort_values(by='전력량', ascending=False)
    
    plt.figure(figsize=(4, 10))
    sns.heatmap(corr_power, annot=True, cmap='coolwarm', center=0)
    plt.title('전력량 상관계수')
    print(f'계약종별: {dt_type[idx]}의 상관관계')
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

ts = df_final.groupby("날짜")["전력량"].mean()   # 날짜별 평균

plt.figure(figsize=(12,4))
plt.plot(ts.index, ts.values, label="월 평균 전력량") 
plt.title("서울시 전력량(월 평균) 추이")
plt.xticks(rotation=45)

plt.legend(loc="upper left")
plt.tight_layout()
plt.show()



In [ ]:
# 그룹화를 통해 날씨 데이터 영향 비교 전처리 

features = ['평균기온','강수량','습도','현지기압','수증기기압','풍속','일조율','적설','지면온도']

weather_type = (
    df_final
    .groupby(["날짜","계약종별"], as_index=False)
    .agg(
        전력량_평균=("전력량","mean"),
        전력량_합계=("전력량","sum"),
        평균기온=("평균기온(°C)","mean"),
        강수량=("월합강수량(00~24h만)(mm)","mean"),
        습도=("평균상대습도(%)","mean"),
        현지기압=("평균현지기압(hPa)","mean"),
        수증기기압=("평균수증기압(hPa)","mean"),
        풍속=("평균풍속(m/s)","mean"),
        일조율=("일조율(%)","mean"),
        적설=("최심적설(cm)","mean"),
        지면온도=("평균지면온도(°C)","mean"),
        HDD=("HDD","mean"),
        CDD=("CDD","mean")
    )
)

weather = (
    df_final
    .groupby(["날짜"], as_index=False)
    .agg(
        전력량_평균=("전력량","mean"),
        전력량_합계=("전력량","sum"),
        평균기온=("평균기온(°C)","mean"),
        강수량=("월합강수량(00~24h만)(mm)","mean"),
        습도=("평균상대습도(%)","mean"),
        현지기압=("평균현지기압(hPa)","mean"),
        수증기기압=("평균수증기압(hPa)","mean"),
        풍속=("평균풍속(m/s)","mean"),
        일조율=("일조율(%)","mean"),
        적설=("최심적설(cm)","mean"),
        지면온도=("평균지면온도(°C)","mean"),
        HDD=("HDD","mean"),
        CDD=("CDD","mean")
    )
)

In [ ]:
# 월별 그룹화를 통한 날씨 데이터와 전력량 상관관계 분석 (합계 평균 결과 같음)
corr = weather[
    ['전력량_합계',"평균기온","강수량","습도",
     "현지기압","수증기기압","풍속","일조율","적설","지면온도","HDD","CDD"]
].corr()

sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("서울시 월별 전력량-날씨 상관관계")
plt.show()


In [ ]:
# 선형 그래프로 날짜 그룹화 + 종별 판단
for idx,name in enumerate(features):
    g = sns.FacetGrid(
        weather_type,
        col="계약종별",
        col_wrap=3,
        height=3,
        sharey=False
    )
    g.map_dataframe(
        sns.regplot,
        x=name,
        y="전력량_합계",
        scatter_kws={"alpha":0.5},
        line_kws={"color":"red"}
    )
    g.fig.suptitle(f"계약종별 {name}-전력량 관계", y=1.03)
    plt.show()

In [ ]:
# pairplot로 나타내기

weather_type = (
    df_final
    .groupby(["날짜","계약종별"], as_index=False)
    .agg(
        전력량_평균=("전력량","mean"),
        전력량_합계=("전력량","sum"),
        평균기온=("평균기온(°C)","mean"),
        강수량=("월합강수량(00~24h만)(mm)","mean"),
        습도=("평균상대습도(%)","mean"),
        현지기압=("평균현지기압(hPa)","mean"),
        수증기기압=("평균수증기압(hPa)","mean"),
        풍속=("평균풍속(m/s)","mean"),
        일조율=("일조율(%)","mean"),
        적설=("최심적설(cm)","mean"),
        지면온도=("평균지면온도(°C)","mean")
    )
)



g = sns.pairplot(weather_type[['전력량_합계','평균기온','강수량','습도','현지기압','수증기기압','풍속','일조율','적설','지면온도']])
g.savefig("pairplot.png", dpi=300, bbox_inches="tight")

In [ ]:
df_final

In [ ]:
df_final.groupby(['계약종별'])['전력량'].mean()

In [ ]:
# 구별 전력량 합계
s = df_final.groupby('시군구')['전력량'].sum().sort_values(ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(s.index, s.values)
plt.title('서울시 구별 전력량 합계')
plt.xlabel('전력량 합계')
plt.ylabel('시군구')
plt.tight_layout()
plt.show()


In [ ]:
# 서울시에서 계약종별의  가장 많은 전력량
s = df_final.groupby('계약종별')['전력량'].mean().sort_values()

plt.Figure(figsize=(8,4))
plt.xlabel('평균 전력량')
plt.title('계약종별 - 평균 전력량')
plt.barh(s.index,s.values)
plt.show()


In [ ]:
df_final

In [ ]:
sns.boxplot(x='계약종별', y='전력량', data=df_final)

In [ ]:
top50 = df_final.nlargest(50, '전력량')[['시군구','계약종별','전력량']]
top50

In [ ]:
df = df_final.copy()

# 1) 월×계약종별로 전력량 합계 (구는 합쳐버림)
ts = df.groupby([df.index, '계약종별'])['전력량'].sum().unstack('계약종별')

# 2) PeriodIndex면 matplotlib용 timestamp로 변환
if isinstance(ts.index, pd.PeriodIndex):
    ts_plot = ts.copy()
    ts_plot.index = ts_plot.index.to_timestamp()
else:
    ts_plot = ts

# 3) 그리기
plt.figure(figsize=(16,6))
for col in ts_plot.columns:
    plt.plot(ts_plot.index, ts_plot[col], marker='o', markersize=2, linewidth=1, label=col)

plt.title('계약종별 월별 전력량 추이(서울 전체)')
plt.xlabel('날짜')
plt.ylabel('전력량')
plt.legend(title='계약종별', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
df = df_final.copy()

# 날짜가 인덱스면 컬럼으로 꺼내기
df = df.reset_index().rename(columns={'index': '날짜'})

# 날짜 타입 정리 (월 단위라 가정)
df['날짜'] = pd.to_datetime(df['날짜'])

# =========================
# 1. 구 × 월 × 계약종별 전력량 집계
# =========================
g = (
    df
    .groupby(['날짜', '시군구', '계약종별'])['전력량']
    .sum()
    .reset_index()
)

# =========================
# 2. 구 × 월 전체 전력량
# =========================
total = (
    g
    .groupby(['날짜', '시군구'])['전력량']
    .sum()
    .reset_index(name='전체전력')
)

# =========================
# 3. 일반용 전력량만 추출
# =========================
general = (
    g[g['계약종별'] == '일반용']
    .rename(columns={'전력량': '일반용전력'})
)

# =========================
# 4. 일반용 비율 계산 (ratio)
# =========================
ratio = general.merge(total, on=['날짜', '시군구'])
ratio['일반용비율'] = ratio['일반용전력'] / ratio['전체전력']

# =========================
# 5. 강남구 vs 기타 구 구분
# =========================
ratio['구분'] = ratio['시군구'].apply(
    lambda x: '강남구' if x == '강남구' else '기타 구'
)

# =========================
# 6. 박스플롯 시각화
# =========================
plt.figure(figsize=(6,4))
sns.boxplot(
    x='구분',
    y='일반용비율',
    data=ratio,
    showfliers=False
)

plt.title('강남구 vs 기타 구 : 일반용 전력 비율')
plt.ylabel('일반용 전력 비율')
plt.xlabel('')
plt.tight_layout()
plt.show()



In [ ]:
# 강남구, 중구, 서초구가 전력량 수요가 많음
plt.figure(figsize=(16,6))
sns.scatterplot(x='시군구', y='전력량', data=df_final)
plt.xticks(rotation=45)
plt.xlabel='서울시 구별'
plt.ylabel='전력량'
plt.show()